In [1]:
from google.colab import files
uploaded = files.upload()

Saving df_test.parquet to df_test.parquet
Saving df_train.parquet to df_train.parquet


In [4]:
import pandas as pd

df_train = pd.read_parquet('df_train.parquet')
df_train.head()

,title,clause_type,question,is_impossible,processed
0,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Document Name,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
1,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Parties,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
2,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Agreement Date,Highlight the parts (if any) of this contract ...,True,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
3,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Effective Date,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."
4,"WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION A...",Expiration Date,Highlight the parts (if any) of this contract ...,False,"{'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1,..."


In [5]:
from transformers import DistilBertTokenizerFast, DistilBertForQuestionAnswering
import torch


tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
model = DistilBertForQuestionAnswering.from_pretrained('distilbert-base-uncased')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
qa_outputs.weight       | MISSING    | 
qa_outputs.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [7]:
device = torch.device('cuda')
model.to(device)

DistilBertForQuestionAnswering(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
     

In [8]:
from torch.utils.data import Dataset

class CUADDataset(Dataset):

  def __init__(self, processed):
    self.processed = processed

  def __len__(self):
    return len(self.processed)

  def __getitem__(self, idx):
      processed = self.processed[idx]
      input_ids = list(processed['input_ids'])[:512]
      attention_mask = list(processed['attention_mask'])[:512]
      input_ids = torch.tensor(input_ids + [0] * (512 - len(input_ids)))
      attention_mask = torch.tensor(attention_mask + [0] * (512 - len(attention_mask)))
      start_position = torch.tensor(processed['start_position'])
      end_position = torch.tensor(processed['end_position'])

      return input_ids, attention_mask, start_position, end_position


In [9]:
train_dataset = CUADDataset(df_train['processed'].values)

In [10]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [11]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

In [12]:
from tqdm import tqdm

epochs = 3
losses = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0
    loop = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch}")
    for batch_idx, (input_ids, attention_mask, start_position, end_position) in loop:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        start_position = start_position.to(device)
        end_position = end_position.to(device)

        outputs = model(input_ids, attention_mask=attention_mask, start_positions=start_position, end_positions=end_position)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        loop.set_postfix(loss=round(loss.item(), 4))

    avg_loss = round(epoch_loss / len(train_loader), 4)
    losses.append(avg_loss)
    print(f"Epoch {epoch} | Avg Loss: {avg_loss}")

Epoch 0: 100%|██████████| 1046/1046 [13:14<00:00,  1.32it/s, loss=0.0119]


Epoch 0 | Avg Loss: 0.4274


Epoch 1: 100%|██████████| 1046/1046 [13:19<00:00,  1.31it/s, loss=0.0179]


Epoch 1 | Avg Loss: 0.1918


Epoch 2: 100%|██████████| 1046/1046 [13:19<00:00,  1.31it/s, loss=0.103]

Epoch 2 | Avg Loss: 0.142


In [13]:
model.save_pretrained('./lexiq_model')
tokenizer.save_pretrained('./lexiq_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./lexiq_model/tokenizer_config.json', './lexiq_model/tokenizer.json')

In [14]:
from google.colab import files
import shutil

shutil.make_archive('lexiq_model', 'zip', './lexiq_model')
files.download('lexiq_model.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>